In [7]:
using Pkg
Pkg.activate(".")

  Activating project at `~/Desktop/Waterloo/Masters/LOO_PIT_Manual/Field_Level_Data`


In [8]:
Pkg.instantiate()
Pkg.resolve()

  No Changes to `~/Desktop/Waterloo/Masters/LOO_PIT_Manual/Field_Level_Data/Project.toml`
  No Changes to `~/Desktop/Waterloo/Masters/LOO_PIT_Manual/Field_Level_Data/Manifest.toml`


In [9]:
# Pkg.add("Turing")
# Pkg.add("Capse")
# Pkg.add("NPZ")
# Pkg.add(url="https://github.com/JuliaCosmologicalLikelihoods/PlanckLite.jl")
# Pkg.add("BenchmarkTools")
# Pkg.add("Plots")
# Pkg.add("Optim")
# Pkg.add("ForwardDiff")
# Pkg.add("LinearAlgebra")
# Pkg.add("Pathfinder")
# Pkg.add("MicroCanonicalHMC")
# Pkg.add("Transducers")
# Pkg.add("StatsPlots")
# Pkg.add("PairPlots")
# Pkg.add("LaTeXStrings")
# Pkg.add("MCMCChains")
# Pkg.add("PSIS")

In [10]:
# using Turing
# using Capse
using NPZ
# # using PlanckLite
# using BenchmarkTools
# using Plots
# using Optim
# using ForwardDiff
# #using LinearAlgebra
# using Pathfinder
# #using MicroCanonicalHMC
# #using Transducers
# using StatsPlots
# #using PairPlots
# #using LaTeXStrings
# #using CairoMakie
# using MCMCChains
using PSIS
# include("utils.jl");

Begin by running PSIS on the full voxels, and storing the results

In [11]:
full_voxel_data = Array{Vector{Float64}}(undef, 32, 32, 32)

for i in 1:32, j in 1:32, k in 1:32
    full_voxel_data[i,j,k] = []
end

In [12]:
for file in 1:8

    x = npzread("/Users/ethansmith/Desktop/Waterloo/Masters/LOO-PIT/loop_it_in_the_field/lpt_32_eh1_ovsamp1/test_with_delta_logdensity_$(file).npz")
    delta_logdensities = x["delta_logdensity"]

    for i in 1:32, j in 1:32, k in 1:32
        full_voxel_data[i, j, k] = append!(full_voxel_data[i, j, k], vec(delta_logdensities[:, :, i, j, k]))
    end
    
end

In [13]:
psis_result = Array{PSISResult{Float64, Vector{Float64}, Int64, Int64, PSIS.GeneralizedPareto{Float64}}}(undef, 32, 32, 32);
for i in 1:32, j in 1:32, k in 1:32
    psis_result[i, j, k] = psis(-full_voxel_data[i,j,k])
end

In [14]:
psis_weights = Array{Vector{Float64}}(undef, 32, 32, 32)
for i in 1:32, j in 1:32, k in 1:32
    psis_weights[i,j,k] = psis_result[i,j,k].weights
end

In [15]:
pareto_shapes = []
for i in 1:32, j in 1:32, k in 1:32
    pareto_shapes = append!(pareto_shapes, psis_result[i,j,k].pareto_shape)
end

Next, use the data to generate Inference Data object

In [16]:
# Pkg.add("ArviZ")
# Pkg.add("ArviZPythonPlots")
# Pkg.add("LinearAlgebra")
# Pkg.add("IrrationalConstants")
# Pkg.add("ArviZExampleData")
# Pkg.add("DimensionalData")
# Pkg.add("Statistics")

In [17]:
using ArviZ
using ArviZPythonPlots
using LinearAlgebra
using IrrationalConstants
using ArviZExampleData
using DimensionalData
using Statistics

In [18]:
idata = InferenceData(; log_densities=namedtuple_to_dataset((; D=full_voxel_data)))

InferenceData with groups:
  > log_densities

In [19]:
idata.log_densities[5,2,17][:D]

8192-element Vector{Float64}:
 7.091101736942965
 4.050110990270539
 3.920632926544473
 4.193935622998795
 4.0754083083278045
 4.143366632495988
 4.10152558324674
 6.554105289437507
 4.764033354591097
 4.2134348848729335
 ⋮
 3.961974708895575
 4.422106042117924
 4.15802761712735
 3.90054043022154
 4.426237167611316
 4.167237114110975
 4.177545560750063
 4.789301300716017
 4.084419328128413

In [20]:
first_element_all_voxels = Vector{Float64}(undef, 32768)

iterator = 1

for i in 1:32
    for j in 1:32
        for k in 1:32
            first_element_all_voxels[iterator] = full_voxel_data[i,j,k][1]
            iterator += 1
        end
    end
end

In [21]:
first_element_all_voxels

32768-element Vector{Float64}:
 4.0696051981190715
 5.4171992675443335
 4.740511732058147
 3.9991714035483463
 5.37952291104228
 4.157540506165735
 4.205445534571416
 4.2836728554292245
 4.8213511688980235
 3.9604017891844445
 ⋮
 4.382361255128817
 4.218577009602887
 4.30792764169778
 4.706306344689559
 4.1689084359898
 4.00948062945153
 3.9601031831052413
 4.067309625529965
 4.280727263188781

In [22]:
# Pkg.add("Distributions")
using Distributions

d = MvNormal(first_element_all_voxels, I)

IsoNormal(
dim: 32768
μ: [4.0696051981190715, 5.4171992675443335, 4.740511732058147, 3.9991714035483463, 5.37952291104228, 4.157540506165735, 4.205445534571416, 4.2836728554292245, 4.8213511688980235, 3.9604017891844445  …  5.557812069249611, 4.382361255128817, 4.218577009602887, 4.30792764169778, 4.706306344689559, 4.1689084359898, 4.00948062945153, 3.9601031831052413, 4.067309625529965, 4.280727263188781]
Σ: [1.0 0.0 … 0.0 0.0; 0.0 1.0 … 0.0 0.0; … ; 0.0 0.0 … 1.0 0.0; 0.0 0.0 … 0.0 1.0]
)


In [23]:
draw = rand(d)

32768-element Vector{Float64}:
 2.205934676101335
 7.404480073850879
 3.5981308209749656
 4.380856739646385
 4.655445538351294
 2.9803951759863643
 4.703313995599166
 4.876406253981463
 4.173682234331094
 4.2113038387765895
 ⋮
 5.094068026894619
 2.6087367252367346
 2.953645029625312
 6.502461942729436
 4.2135571471932
 4.340623803451965
 2.6378787314214147
 3.727857450015955
 4.421744016610602

In [24]:
draws = Vector{Array{Float64}}(undef, 8192)

for n in 1:8192
    new_µ_all_voxels = Vector{Float64}(undef, 32768)

    iterator = 1
    for i in 1:32
        for j in 1:32
            for k in 1:32
                new_µ_all_voxels[iterator] = full_voxel_data[i,j,k][n]
                iterator += 1
            end
        end
    end
    dist = MvNormal(new_µ_all_voxels, I)
    draws[n] = rand(dist)
end

In [33]:
draws[1]

32768-element Vector{Float64}:
 3.456644168501203
 3.959513601772914
 4.469319343878912
 3.6460279408975955
 6.982392879373728
 4.204543922959497
 4.466013162950964
 4.6122626473848705
 5.289727787794211
 4.656704359865771
 ⋮
 4.987315709485454
 4.737766182265105
 4.7358686791726665
 4.82135429974068
 6.145527837889181
 4.504562223479913
 3.791286941229184
 2.2869139676565746
 4.3484656723403425

In [ ]:
draws_voxels = Vector{Array{}}

In [26]:
a = [1,2,3]
dist = MvNormal(a, I)
rand(dist)

3-element Vector{Float64}:
 -0.21831159668031064
  1.9893860855262553
  4.3824845036349345

In [27]:
b = Array{Float64}(undef, 3, 3, 3)
for i in 1:3
    for j in 1:3
        for k in 1:3
            b[i,j,k] = i + j*2 + k*3
        end
    end
end
b

3×3×3 Array{Float64, 3}:
[:, :, 1] =
 6.0   8.0  10.0
 7.0   9.0  11.0
 8.0  10.0  12.0

[:, :, 2] =
  9.0  11.0  13.0
 10.0  12.0  14.0
 11.0  13.0  15.0

[:, :, 3] =
 12.0  14.0  16.0
 13.0  15.0  17.0
 14.0  16.0  18.0

In [28]:
new_b = Array{Float64}(undef, 3, 3, 3)
v = vec(b)
for i in 1:3
    for k in 1:3
        for j in 1:3
            new_b[i,j,k] = v[(k-1)*9+(j-1)*3+i]
        end
    end
end
    

In [29]:
new_b

3×3×3 Array{Float64, 3}:
[:, :, 1] =
 6.0   8.0  10.0
 7.0   9.0  11.0
 8.0  10.0  12.0

[:, :, 2] =
  9.0  11.0  13.0
 10.0  12.0  14.0
 11.0  13.0  15.0

[:, :, 3] =
 12.0  14.0  16.0
 13.0  15.0  17.0
 14.0  16.0  18.0

In [36]:
draws_voxels = Vector{Array{Float64}}(undef, 3)

3-element Vector{Array{Float64}}:
 #undef
 #undef
 #undef